In [7]:
from langchain_core.output_parsers import StrOutputParser
# Purpose: extract the .content field from an AIMessage and return it as str

from langchain_core.prompts import PromptTemplate

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
model = ChatOpenAI(model='gpt-4.1-nano')
prompt_template = PromptTemplate.from_template("What is the capital of {country}?")
chain = prompt_template | model | StrOutputParser()

res = chain.invoke({'country': 'France'})
print(res)

The capital of France is Paris.


Chat models can utilize lists of messages (e.g. system messages). We can create templates that prepare a list of messages.

In [14]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.messages import SystemMessage, HumanMessage

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful assistant that answers questions about geography."
                  " If they ask about a country, you should add one fun fact about it."),
    HumanMessagePromptTemplate.from_template("What is the capital of {country}?")
])
prompt.invoke({'country': 'France'})

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant that answers questions about geography. If they ask about a country, you should add one fun fact about it.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={})])

In [15]:
chain = prompt | model | StrOutputParser()
res = chain.invoke({'country': 'France'})
print(res)

The capital of France is Paris. Fun fact: Paris is often called "The City of Light" because it was one of the first cities in the world to have street lighting.


In [16]:
# A fast and convenient way to build message prompts:
chat_prompt_template = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant who is an expert in math'),
    ('human', 'What is 2+2')
])
chat_prompt_template.invoke({})

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant who is an expert in math', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={})])

The Gemini prompting guide recommends that each prompt should have four parts:
- persona
- task
- relevant context
- desired format

In [19]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("arietem/math_cot")

In [27]:
prompt.input_variables

['question']

In [26]:
cot_chain = prompt | model | StrOutputParser()
res = cot_chain.invoke({'question': 'What is 12*8?'})
print(res)

<answer>96</answer>


### Chain-of-thought + Concise response

CoT is powerful and has been demonstrated multiple times that it significantly increases performance in many cases. 

In [29]:
from operator import itemgetter

parse_prompt_template = (
    "Given the initial question and a full answer, "
    "extract the concise answer. Do not assume anything and "
    "only use a provided full answer.\n\nQUESTION:\n{question}\n"
    "FULL ANSWER:\n{full_answer}\n\nCONCISE ANSWER:\n"
)
parse_prompt = PromptTemplate.from_template(
    parse_prompt_template
)

final_chain = (
    {
        "full_answer": itemgetter("question") | cot_chain,
        "question": itemgetter("question"),
    }
    | parse_prompt
    | model
    | StrOutputParser()
)
res = final_chain.invoke({"question": "What is 12*8?"})
print(res)

96
